In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# -*- coding: utf-8 -*-
"""
ASR Agent 1: WAv2Vec2-XL-R-300M-FINETUNE

This version
"""

# 1. Install and Import Necessary Libraries
# -------------------------------------------
# Using the same libraries as the first agent for consistency.
!pip install -q transformers accelerate torchaudio librosa
!pip install torchcodec==0.2
!pip install fsspec==2023.9.2
!pip install datasets==2.10.0
!pip install evaluate==0.4.0
!pip install jiwer==2.3.0
!pip install tf-keras

!pip install --user --no-cache-dir pyarrow==15.0.2

import os
import re
import json
from dataclasses import dataclass
from typing import Dict, List, Optional, Union

import torch
import torch.nn as nn
import torchaudio
import numpy as np
from torch.nn.modules.utils import _pair # Needed for custom Conv2d

from datasets import load_dataset, Audio, DatasetDict
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer
)
import evaluate


  Using cached datasets-2.10.0-py3-none-any.whl.metadata (20 kB)
Using cached datasets-2.10.0-py3-none-any.whl (469 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
# 2. Configuration Variables
# --------------------------
# Dataset related (UC-DM-01)
DATASET_NAME = "aconeil/nchlt"
DEFAULT_CONFIG_NAME = "default"
TEXT_COLUMN = "transcription"
AUDIO_COLUMN = "audio"

# Model related (UC-AD-01)
MODEL_NAME = "facebook/wav2vec2-xls-r-300m"
MODEL_OUTPUT_DIR = "/content/drive/MyDrive/wav2vec2-finetune-checkpoints22"
TOKENIZER_OUTPUT_DIR = "/content/drive/MyDrive/wav2vec2-xls-r-300m-finetune-tokenizer22" # for vocabulary

# Training parameters (UC-AD-02)
TARGET_SAMPLING_RATE = 16_000
MAX_DURATION_IN_SECONDS = 20.0 # Filter out very long audio samples to avoid OOM errors
MIN_DURATION_IN_SECONDS = 1.0  # Filter out very short audio samples
NUM_EPOCHS = 10 # Start with a few epochs for prototyping
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2 # Effective batch size = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
LEARNING_RATE = 3e-4 # Often a good starting point for Wav2Vec2 fine-tuning
WARMUP_STEPS = 500
SAVE_STEPS = 1000
EVAL_STEPS = 1000


In [ ]:
# Forcefully clear Hugging Face datasets and general cache
!rm -rf ~/.cache/huggingface/datasets/*
!rm -rf ~/.cache/huggingface/modules/*
!rm -rf ~/.cache/huggingface/hub/*
print("Hugging Face caches cleared.")

Hugging Face caches cleared.


In [ ]:
%pip install -U datasets

# 3. Load and Prepare Dataset (UC-DM-01)
# -------------------------------------------------
print(f"Loading dataset: {DATASET_NAME}, config: {DEFAULT_CONFIG_NAME}")
try:
    raw_datasets = DatasetDict()
    # Load train and test splits
    train_ds = load_dataset(DATASET_NAME, name=DEFAULT_CONFIG_NAME, split="train")
    test_ds = load_dataset(DATASET_NAME, name=DEFAULT_CONFIG_NAME, split="test")

    # This dataset doesn't have a pre-defined validation split.
    # We will create one from the training data.
    print("Train and test splits loaded. Creating validation split from training data...")
    # Split the training data (90% train, 10% validation)
    train_val_split = train_ds.train_test_split(test_size=0.1, shuffle=True, seed=42) # Use a seed for reproducibility

    raw_datasets["train"] = train_val_split["train"]
    raw_datasets["validation"] = train_val_split["test"] # The 'test' part of this split becomes our validation set
    raw_datasets["test"] = test_ds # Keep the original test set

    print("Dataset loaded and split successfully.")
    print(raw_datasets)
    print(f"\nFeatures of the training set: {raw_datasets['train'].features}")
    if len(raw_datasets['train']) > 0:
        print(f"First example from the training set: {raw_datasets['train'][0]}")

except Exception as e:
    print(f"Error loading dataset: {e}")
    from datasets import get_dataset_config_names
    try:
        configs = get_dataset_config_names(DATASET_NAME)
        print(f"Available configs for {DATASET_NAME}: {configs}")
    except Exception as e_configs:
        print(f"Could not retrieve configs for {DATASET_NAME}: {e_configs}")
    raise e


  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
Using cached datasets-4.0.0-py3-none-any.whl (494 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 2.10.0
    Uninstalling datasets-2.10.0:
      Successfully uninstalled datasets-2.10.0


Loading dataset: aconeil/nchlt, config: default


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/41871 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2802 [00:00<?, ? examples/s]

Dataset parquet downloaded and prepared to /root/.cache/huggingface/datasets/aconeil___parquet/aconeil--nchlt-be2e393fe3a31823/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec. Subsequent calls will reuse this data.


/usr/local/lib/python3.11/dist-packages/datasets/table.py:1427: FutureWarning: promote has been superseded by promote_options='default'.
  we modify both to have 4 row_blocks of size 2, 1, 1 and 2:


Train and test splits loaded. Creating validation split from training data...
Dataset loaded and split successfully.
DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'gender', 'age', 'speaker_id', 'duration', 'md5sum', 'pdp_score'],
        num_rows: 37683
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'gender', 'age', 'speaker_id', 'duration', 'md5sum', 'pdp_score'],
        num_rows: 4188
    })
    test: Dataset({
        features: ['audio', 'transcription', 'gender', 'age', 'speaker_id', 'duration', 'md5sum', 'pdp_score'],
        num_rows: 2802
    })
})

Features of the training set: {'audio': Audio(sampling_rate=None, mono=True, decode=True, id=None), 'transcription': Value(dtype='string', id=None), 'gender': Value(dtype='string', id=None), 'age': Value(dtype='int64', id=None), 'speaker_id': Value(dtype='int64', id=None), 'duration': Value(dtype='float64', id=None), 'md5sum': Value(dtype='string', id=None), 'pdp_score

In [ ]:
# --- Text Preprocessing --- (UC-DM-02)
# Remove special characters, keep isiZulu characters and spaces
chars_to_remove_regex = r'[!"#$%&\'()*+,-./:;<=>?@\[\\\]^_`{|}~0-9]'

def remove_special_characters(batch):
    # Ensure the text column exists and is not None
    if batch[TEXT_COLUMN] is not None:
        batch[TEXT_COLUMN] = re.sub(chars_to_remove_regex, '', batch[TEXT_COLUMN]).lower()
    else:
        # Handle cases where the text might be missing, e.g., by providing an empty string
        batch[TEXT_COLUMN] = ""
    return batch

print("Normalizing text data...")
raw_datasets = raw_datasets.map(remove_special_characters, num_proc=os.cpu_count())

Normalizing text data...


Map (num_proc=2):   0%|          | 0/37683 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1427: FutureWarning: promote has been superseded by promote_options='default'.
  we modify both to have 4 row_blocks of size 2, 1, 1 and 2:


Map (num_proc=2):   0%|          | 0/4188 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":


Map (num_proc=2):   0%|          | 0/2802 [00:00<?, ? examples/s]

In [ ]:
# --- Create Vocabulary ---
# For Mapping each unique character present in the text to a numerical ID

#def extract_all_chars(batch):
  #all_text = " ".join(batch[TEXT_COLUMN])
  #vocab = list(set(all_text))
  #return {"vocab": [vocab], "all_text": [all_text]}

print("Extracting vocabulary...")
# Concatenate all splits to build a comprehensive vocabulary
combined_text_data = "".join([text for split in raw_datasets for text in raw_datasets[split][TEXT_COLUMN] if text is not None])
vocab_list = list(set(combined_text_data))
vocab_dict = {v: k for k, v in enumerate(vocab_list)}
vocab_dict["|"] = vocab_dict[" "] # CTC blank token representation
del vocab_dict[" "]
vocab_dict["[UNK]"] = len(vocab_dict) # Unknown token
vocab_dict["[PAD]"] = len(vocab_dict) # Padding token for tokenizer (CTC doesn't use it for loss)
print(f"Vocabulary size: {len(vocab_dict)}")


Extracting vocabulary...
Vocabulary size: 29


In [ ]:
# Save vocabulary
os.makedirs(TOKENIZER_OUTPUT_DIR, exist_ok=True)
with open(os.path.join(TOKENIZER_OUTPUT_DIR, 'vocab.json'), 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

In [ ]:
# --- Load Tokenizer and Feature Extractor ---
# Set up the necessary tools, a tokenizer and a feature extractor, to prepare both the text transcripts and the audio data for the Wav2Vec2 model.
# (UC-AD-01)
tokenizer = Wav2Vec2CTCTokenizer(
    os.path.join(TOKENIZER_OUTPUT_DIR, 'vocab.json'),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=TARGET_SAMPLING_RATE,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=False # CTC doesn't use attention mask for features
)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained(MODEL_OUTPUT_DIR) # Save processor with the model


[]

In [ ]:
# --- Audio Preprocessing ---
# (UC-DM-02)
# Ensure audio is 16kHz mono
print("Casting audio column to 16kHz Audio feature...")
raw_datasets = raw_datasets.cast_column(
    AUDIO_COLUMN, Audio(sampling_rate=TARGET_SAMPLING_RATE)
)

Casting audio column to 16kHz Audio feature...


In [ ]:
# Filter audio by duration
def filter_duration(example):
    if example[AUDIO_COLUMN] is None or example[AUDIO_COLUMN]["array"] is None:
        return False
    duration = len(example[AUDIO_COLUMN]["array"]) / example[AUDIO_COLUMN]["sampling_rate"]
    return MIN_DURATION_IN_SECONDS <= duration <= MAX_DURATION_IN_SECONDS

print(f"Filtering audio by duration ({MIN_DURATION_IN_SECONDS}s - {MAX_DURATION_IN_SECONDS}s)...")
initial_sizes = {split: len(raw_datasets[split]) for split in raw_datasets}
raw_datasets = raw_datasets.filter(filter_duration, num_proc=os.cpu_count())
filtered_sizes = {split: len(raw_datasets[split]) for split in raw_datasets}

for split in initial_sizes:
    print(f"Split '{split}': {initial_sizes[split]} -> {filtered_sizes[split]} samples after duration filtering.")



Filtering audio by duration (1.0s - 20.0s)...


Filter (num_proc=2):   0%|          | 0/37683 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":


Filter (num_proc=2):   0%|          | 0/4188 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":


Filter (num_proc=2):   0%|          | 0/2802 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":


Split 'train': 37683 -> 37667 samples after duration filtering.
Split 'validation': 4188 -> 4186 samples after duration filtering.
Split 'test': 2802 -> 2802 samples after duration filtering.


In [ ]:
#Transform the raw audio and text data into a format that can be understood and processed by the Wav2Vec2 model.

def prepare_dataset(batch):
    audio = batch[AUDIO_COLUMN]
    # batched output is "un-batched"
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["input_length"] = len(batch["input_values"])

    with processor.as_target_processor():
        batch["labels"] = processor(batch[TEXT_COLUMN]).input_ids
    return batch

print("Preparing dataset with processor...")
processed_datasets = raw_datasets.map(
    prepare_dataset,
    remove_columns=raw_datasets.column_names["train"], # remove original columns
    num_proc=os.cpu_count() # use num_proc for faster processing
)
print("Dataset preparation complete.")


Preparing dataset with processor...


Map (num_proc=2):   0%|          | 0/37667 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be 

Map (num_proc=2):   0%|          | 0/4186 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be 

Map (num_proc=2):   0%|          | 0/2802 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  def from_tables(cls, tables: list[Union[pa.Table, Table]], axis: int = 0) -> "ConcatenationTable":
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be 

Dataset preparation complete.


In [ ]:
# 4. Load Pre-trained Model (UC-AD-01)
# --------------------------------------
print(f"Loading pre-trained model: {MODEL_NAME}")
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ctc_loss_reduction="mean", # "sum" can also be used
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer)
)

model.freeze_feature_extractor()

Loading pre-trained model: facebook/wav2vec2-xls-r-300m


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-300m and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/modeling_wav2vec2.py:1823: FutureWarning: The method `freeze_feature_extractor` is deprecated and will be removed in Transformers v5. Please use the equivalent `freeze_feature_encoder` method instead.
  warnings.warn(


In [ ]:
# 9. Test the model with a single example
# ---------------------------------------
print("Testing the model with a single example from the test set...")

# Select a single example from the test dataset
sample = processed_datasets["test"][0]

# Get the input values and labels
input_values = torch.tensor(sample["input_values"]).unsqueeze(0)
labels = torch.tensor(sample["labels"]).unsqueeze(0)

# Move the model to evaluation mode and to the appropriate device
model.eval()
if torch.cuda.is_available():
    model.to("cuda")
    input_values = input_values.to("cuda")
    labels = labels.to("cuda")


# Perform inference
with torch.no_grad():
    logits = model(input_values).logits

# Get the predicted token IDs
predicted_ids = torch.argmax(logits, dim=-1)

# Decode the predicted token IDs to text
predicted_sentence = processor.batch_decode(predicted_ids)[0]

# Decode the ground truth labels to text
ground_truth_sentence = processor.batch_decode(labels, group_tokens=False)[0]

print(f"Ground Truth: {ground_truth_sentence}")
print(f"Predicted: {predicted_sentence}")

Testing the model with a single example from the test set...
Ground Truth: izokwenza umyalelo wokuthi
Predicted: b brib brbib brbi b b b</s>b b b</s> b bib bibaicab b


In [ ]:
RAW_TEXT_COLUMN = "text"

# The dataset split you want to check (e.g., "train", "test")
SPLIT = "test"
# --------------------
import random
def verify_random_sample(raw_datasets, processed_datasets, processor):
    """
    Selects a random sample and compares its raw text to its processed label.
    """
    print(f"--- Verifying a random sample from the '{SPLIT}' split ---")

    # 1. Select a random index
    try:
        num_samples = len(processed_datasets[SPLIT])
        random_index = random.randint(0, num_samples - 1)
        print(f"Checking sample at index: {random_index}")
    except Exception as e:
        print(f"Could not get random sample. Error: {e}")
        return

    # 2. Get the corresponding samples from both datasets
    processed_sample = processed_datasets[SPLIT][random_index]
    raw_sample = raw_datasets[SPLIT][random_index]

    # 3. Get the original text from the raw dataset
    try:
        original_text = raw_sample[TEXT_COLUMN]
    except KeyError:
        print(f"ERROR: Column '{TEXT_COLUMN}' not found in the raw dataset.")
        print(f"Available columns are: {list(raw_sample.keys())}")
        return

    # 4. Decode the processed labels back to text
    labels = torch.tensor(processed_sample["labels"]).unsqueeze(0)
    decoded_processed_text = processor.batch_decode(labels, group_tokens=False)[0]

    # 5. Print for comparison
    print("\nOriginal Text from Raw File:")
    print(f"==> \"{original_text}\"")

    print("\nProcessed Label (Decoded):")
    print(f"==> \"{decoded_processed_text}\"")


# --- Run the verification ---
# Make sure your datasets and processor are loaded before calling this
verify_random_sample(raw_datasets, processed_datasets, processor)

--- Verifying a random sample from the 'test' split ---
Checking sample at index: 10

Original Text from Raw File:
==> "uma zikhuluma nabantu"

Processed Label (Decoded):
==> "uma zikhuluma nabantu"


In [ ]:
# 5. Define Training Components (UC-AD-02)
# ----------------------------------------
# Data Collator
# This is responsible for taking a list of individual data samples and combining them into a batch that can be fed into the model during training.
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lenghts and need different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [ ]:
# Evaluation Metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    # replace -100 with pad_token_id
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)

    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)   # we do not want to group tokens when computing the metrics

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}


In [ ]:
# Training Arguments
training_args = TrainingArguments(
  output_dir=MODEL_OUTPUT_DIR,
  group_by_length=True, # Speeds up training by batching similar length audio
  per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
  per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
  gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
  eval_strategy="steps",
  num_train_epochs=NUM_EPOCHS,
  fp16=True,
  save_steps=SAVE_STEPS,
  eval_steps=EVAL_STEPS,
  logging_steps=EVAL_STEPS // 2,
  learning_rate=LEARNING_RATE,
  warmup_steps=WARMUP_STEPS,
  save_total_limit=2,
  load_best_model_at_end=True,
  metric_for_best_model="wer",
  greater_is_better=False,
)


In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["validation"],
    tokenizer=processor.feature_extractor, #pass the feature_extractor for Wav2Vec2
)


/tmp/ipython-input-1774896493.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# 6. Start Fine-tuning (UC-AD-02)
# --------------------------------
print("Starting fine-tuning...")
try:
    trainer.train()
    print("Fine-tuning finished.")
except Exception as e:
    print(f"Error during training: {e}")
    print("This might be due to OOM errors. Try reducing batch size or gradient accumulation steps.")
    raise e

# API KEY - 570d6a37ed4ae6a545d580fb486ed630471604a7

Starting fine-tuning...


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lesego-mogable (lesego-mogable-university-of-johannesburg) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


Step,Training Loss,Validation Loss,Wer,Cer
1000,1.356700,0.529297,0.778241,0.160475
2000,0.544700,0.322137,0.635693,0.115177
3000,0.446600,0.286011,0.605628,0.106234
4000,0.404100,0.249668,0.581817,0.097959
5000,0.370800,0.230306,0.553115,0.092295
6000,0.346200,0.212839,0.541249,0.089094
7000,0.331900,0.202961,0.533633,0.085780
8000,0.295800,0.186800,0.522649,0.082814
9000,0.281600,0.181951,0.513750,0.079691
10000,0.255800,0.173856,0.504049,0.077618


/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call

KeyboardInterrupt: 

In [ ]:
# 7. Save the final model and tokenizer
# ---------------------------------------
print("Saving the final trained model and tokenizer...")
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print(f"Model and tokenizer saved to {MODEL_OUTPUT_DIR}")

Saving the final trained model and tokenizer...
Model and tokenizer saved to /content/drive/MyDrive/wav2vec2-finetune-checkpoints22


In [ ]:
import torch
import torchaudio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

# 1. Define the path to your saved model
model_path = "/content/drive/MyDrive/wav2vec2-finetune-checkpoints"

# 2. Load the fine-tuned model and processor
model = Wav2Vec2ForCTC.from_pretrained(model_path)
processor = Wav2Vec2Processor.from_pretrained(model_path)

# 3. Move the model to a GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model and processor loaded successfully.")

In [ ]:
# Ensure the trainer has been initialized before running this cell.

print("Evaluating the model on the test dataset...")

# The trainer.evaluate() method runs inference on the provided dataset
# and computes the metrics you defined in compute_metrics.
test_results = trainer.evaluate(eval_dataset=processed_datasets["test"])

print("\n--- Test Set Evaluation Results ---")
print(f"WER: {test_results['eval_wer']:.4f}")
print(f"CER: {test_results['eval_cer']:.4f}")
print("\nComplete results dictionary:")
print(test_results)

Evaluating the model on the test dataset...


Step,Training Loss,Validation Loss,Wer,Cer
1000,1.356700,0.529297,0.778241,0.160475
2000,0.544700,0.322137,0.635693,0.115177
3000,0.446600,0.286011,0.605628,0.106234
4000,0.404100,0.249668,0.581817,0.097959
5000,0.370800,0.230306,0.553115,0.092295
6000,0.346200,0.212839,0.541249,0.089094
7000,0.331900,0.202961,0.533633,0.085780
8000,0.295800,0.186800,0.522649,0.082814
9000,0.281600,0.181951,0.513750,0.079691
10000,0.255800,0.173856,0.504049,0.077618



--- Test Set Evaluation Results ---
WER: 0.4660
CER: 0.0663

Complete results dictionary:
{'eval_loss': 0.1257307529449463, 'eval_wer': 0.46604353999053477, 'eval_cer': 0.066320293398533}


In [ ]:
import numpy as np

# Let's predict the first 5 examples from the test set
num_examples = 5
test_subset = processed_datasets["test"].select(range(num_examples))

print(f"Running prediction on the first {num_examples} test samples...")

# Get predictions from the trainer
predictions_output = trainer.predict(test_subset)

# The output contains predictions (logits) and label_ids (ground truth)
predicted_ids = np.argmax(predictions_output.predictions, axis=-1)
true_label_ids = predictions_output.label_ids

# Decode both the predicted and true labels
# For labels, we need to replace -100 (the padding token) before decoding
true_label_ids[true_label_ids == -100] = processor.tokenizer.pad_token_id
predicted_transcriptions = processor.batch_decode(predicted_ids)
true_transcriptions = processor.batch_decode(true_label_ids, group_tokens=False)

print("\n--- Prediction Samples ---")
for i in range(num_examples):
    print(f"Sample {i+1}:")
    print(f"  Ground Truth: {true_transcriptions[i]}")
    print(f"  Predicted:    {predicted_transcriptions[i]}")
    print("-" * 20)

Running prediction on the first 5 test samples...


Step,Training Loss,Validation Loss,Wer,Cer
1000,1.356700,0.529297,0.778241,0.160475
2000,0.544700,0.322137,0.635693,0.115177
3000,0.446600,0.286011,0.605628,0.106234
4000,0.404100,0.249668,0.581817,0.097959
5000,0.370800,0.230306,0.553115,0.092295
6000,0.346200,0.212839,0.541249,0.089094
7000,0.331900,0.202961,0.533633,0.085780
8000,0.295800,0.186800,0.522649,0.082814
9000,0.281600,0.181951,0.513750,0.079691
10000,0.255800,0.173856,0.504049,0.077618



--- Prediction Samples ---
Sample 1:
  Ground Truth: liphungulwe uma lokhu
  Predicted:    liphungulwe uma lokhu
--------------------
Sample 2:
  Ground Truth: kungathatha isikhathi eside
  Predicted:    kungathatha isikhathi eside
--------------------
Sample 3:
  Ground Truth: lenqubo ibonakala sengathi
  Predicted:    lenqubo ibonakala sengathi
--------------------
Sample 4:
  Ground Truth: lesi siphandla sinikezwa
  Predicted:    lesi siphandla sinikezwa
--------------------
Sample 5:
  Ground Truth: izokwenza umyalelo wokuthi
  Predicted:    izokwenza umyalelo wokuthi
--------------------
